# Caderno de Comparação Qualitativa de Modelos (NLG)

**Objetivo:** Este notebook executa um teste comparativo entre diferentes modelos da família Gemma para avaliar sua capacidade de seguir as instruções dos prompts pedagógicos do agente LeIA. O objetivo é analisar qualitativamente as respostas geradas para um conjunto fixo de perguntas, uma para cada intenção pedagógica, e selecionar o modelo que oferece o melhor equilíbrio entre performance, leveza e alinhamento filosófico com o projeto.

In [ ]:
import sys
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import textwrap

# --- Configuração de Caminho ---
ROOT_DIR = Path.cwd().parent.parent 
sys.path.append(str(ROOT_DIR))
from src.classifier.classifier import NLUClassifier

print("Ambiente configurado com sucesso.")

Ambiente configurado com sucesso.


In [3]:
# --- Carregar o Módulo NLU (será usado em todos os testes) ---
MODELO_NLU_FINAL = ROOT_DIR / "models" / "leia_classifier_1k_final"
nlu_classifier = NLUClassifier(model_path=str(MODELO_NLU_FINAL))

# --- Lista de Modelos NLG a Serem Testados ---
models_to_test = [
    "google/gemma-2-2b-it",
    "google/gemma-3-1b-it",
    "google/gemma-3-4b-it"
]

# --- Lista de Perguntas de Teste (uma por intenção) ---
test_questions = [
    {
        "intent": "Conceitual",
        "discipline": "História",
        "question": "o que foi o Renascimento e quais suas principais características?"
    },
    {
        "intent": "Procedimental", # Sabemos que o NLU pode errar, mas o prompt correto será usado
        "discipline": "Química",
        "question": "como posso balancear a equação química Fe + O2 -> Fe2O3?"
    },
    {
        "intent": "Análise de Exemplo",
        "discipline": "Geografia",
        "question": "Com base no texto sobre a urbanização brasileira, quais foram os principais fatores que levaram ao crescimento desordenado das cidades?"
    },
    {
        "intent": "Comparativo",
        "discipline": "Ciências Humanas",
        "question": "Qual a principal diferença entre os sistemas presidencialista e parlamentarista?"
    }
]

print("Modelos e perguntas de teste definidos.")

Device set to use mps:0


Modelo NLU carregado com sucesso de '/Users/giossaurus/Developer/leia_tcc/models/leia_classifier_1k_final'
Modelos e perguntas de teste definidos.


/opt/homebrew/Caskroom/miniforge/base/envs/torch-mps/lib/python3.10/site-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [ ]:
def get_freirian_response(nlu_classifier, nlg_model_name, user_question, discipline):
    print(f"--- Carregando modelo NLG: {nlg_model_name} ---")
    nlg_tokenizer = AutoTokenizer.from_pretrained(nlg_model_name)
    nlg_model = AutoModelForCausalLM.from_pretrained(nlg_model_name)
    nlg_pipeline = pipeline(
        "text-generation",
        model=nlg_model,
        tokenizer=nlg_tokenizer,
        max_new_tokens=200,
    )
    
    # 1. Obter a intenção (usamos o NLU real)
    nlu_result = nlu_classifier.predict(user_question)
    intent = nlu_result['label']
    print(f"Intenção detectada pelo NLU: {intent}")

    # 2. Carregar e selecionar o template de prompt
    templates = {
        "Conceitual": """
**PERSONA:** Você é um especialista em Paulo Freire, atuando como o arquiteto pedagógico do agente de IA LeIA. Sua tarefa é gerar uma resposta que materialize a filosofia do projeto.
**CONTEXTO:** A pergunta é sobre {disciplina}. A intenção é "Conceitual".
**Pergunta do Aluno:** "{user_question}"
**TAREFA:** Com base nos conceitos de "educação bancária" vs. "educação problematizadora", gere uma resposta que:
1.  **Inicie a Dialogicidade:** Comece validando a pergunta, mas imediatamente a transforme em um problema a ser investigado.
2.  **Combata a "Descarga Cognitiva":** Evite "depositar" o conhecimento. Em vez disso, conecte o conceito a um "tema gerador" da realidade do aluno.
3.  **Implemente o Scaffolding:** Crie "andaimes pedagógicos" através de uma ou duas perguntas-guia que incentivem o "pensar autêntico".
4.  **Materialize o Artefato Tecnopolítico:** A resposta deve refletir a intencionalidade do LeIA como uma ferramenta de resistência à passividade.
""",
        "Procedimental": """
**PERSONA:** Você é um educador freiriano e especialista em PLN, responsável por treinar a "persona" do agente LeIA.
**CONTEXTO:** A pergunta é sobre {disciplina}. A intenção é "Procedimental".
**Pergunta do Aluno:** "{user_question}"
**TAREFA:** Fundamentado no conceito de "práxis" (unidade indissociável entre ação e reflexão), gere uma resposta que:
1.  **Valide a Ação, Inicie a Reflexão:** Reconheça a necessidade de aplicar um procedimento, mas questione o propósito e a origem desse método.
2.  **Evite a Resposta Direta:** Não entregue o passo a passo completo. Forneça um "andaime" inicial (o primeiro passo ou o conceito-chave).
3.  **Promova a Autonomia:** Incentive a experimentação, posicionando o aluno como um agente ativo.
""",
        "Análise de Exemplo": """
**PERSONA:** Você é um especialista na obra de Paulo Freire e na análise sociotécnica, projetando o comportamento dialógico do LeIA.
**CONTEXTO:** A pergunta é sobre {disciplina}. A intenção é "Análise de Exemplo".
**Pergunta do Aluno:** "{user_question}"
**TAREFA:** Gere uma resposta que transforme a tarefa de interpretação em um ato de "conscientização". A resposta deve:
1.  **Focar no Sujeito Leitor:** Em vez de interpretar o exemplo para o aluno, guie-o a se tornar o sujeito da interpretação (ex: "O que mais chamou sua atenção nesse trecho?").
2.  **Conectar Exemplo e Realidade:** Utilize o exemplo como uma "ponte" para o mundo vivido pelo aluno.
3.  **Construir o Andaime Interpretativo (Scaffolding):** Ofereça uma pista conceitual ou uma pergunta direcionada sem entregar a análise final.
""",
        "Comparativo": """
**PERSONA:** Você é um especialista na obra de Paulo Freire, atuando como o arquiteto pedagógico do LeIA.
**CONTEXTO:** A pergunta é sobre {disciplina}. A intenção é "Comparativo".
**Pergunta do Aluno:** "{user_question}"
**TAREFA:** Com base no conceito de "educação problematizadora", gere uma resposta que:
1.  **Valide e Aprofunde a Questão:** Pergunte ao aluno o que ele já entende sobre cada um dos pontos separadamente.
2.  **Fornecer um Eixo de Análise (Scaffolding):** Sugira um critério ou um ponto de vista a partir do qual a comparação pode começar.
3.  **Incentivar a Síntese, não a Separação:** Formule uma pergunta final que guie o aluno a tirar uma conclusão.
"""
    }
    prompt_template = templates.get(intent, templates["Conceitual"])
    
    # 3. Formatar o prompt final
    final_prompt = prompt_template.format(disciplina=discipline, user_question=user_question)
    
    # 4. Gerar a resposta com o modelo NLG
    messages = [{"role": "user", "content": final_prompt}]
    prompt_formatted = nlg_pipeline.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    outputs = nlg_pipeline(
        prompt_formatted, do_sample=True, temperature=0.7, top_k=50, top_p=0.95
    )
    generated_text = outputs[0]["generated_text"]
    response = generated_text[len(prompt_formatted):].strip()
    
    return response

print("Função de teste 'get_freirian_response' definida com templates completos do LeIA.")

Função de teste 'get_freirian_response' definida com templates completos do LeIA.


In [ ]:
# --- Executar os Testes e Imprimir os Resultados ---
results = {}

for model_name in models_to_test:
    print(f"==========================================================")
    print(f"INICIANDO TESTES COM O MODELO: {model_name}")
    print(f"==========================================================")
    
    results[model_name] = []
    
    for test in test_questions:
        try:
            response = get_freirian_response(
                nlu_classifier=nlu_classifier,
                nlg_model_name=model_name,
                user_question=test["question"],
                discipline=test["discipline"]
            )
            
            results[model_name].append({
                "question": test["question"],
                "intent": test["intent"],
                "response": response
            })
            
            print("\n" + "="*50)
            print(f"MODELO: {model_name}")
            print(f"PERGUNTA ({test['intent']}): {test['question']}")
            print("-" * 20)
            print("RESPOSTA GERADA:")
            print(textwrap.fill(response, width=80)) # Formata o texto para não quebrar a tela
            print("="*50 + "\n")

        except Exception as e:
            print(f"\nERRO ao testar o modelo {model_name} com a pergunta '{test['question']}': {e}\n")

print("Todos os testes foram concluídos.")

INICIANDO TESTES COM O MODELO: google/gemma-2-2b-it
--- Carregando modelo NLG: google/gemma-2-2b-it ---


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use mps:0


Intenção detectada pelo NLU: Conceitual

MODELO: google/gemma-2-2b-it
PERGUNTA (Conceitual): o que foi o Renascimento e quais suas principais características?
--------------------
RESPOSTA GERADA:
A pergunta sobre o Renascimento é um ponto de partida excelente para um mergulho
profundo na complexidade da história, e sua importância para o nosso presente!
É, de fato, um período que marcou um antes e um depois na forma como a sociedade
se organizava, pensava e aprendia. Mas, ao invés de apenas te apresentar uma
visão superficial do que foi o Renascimento, vamos juntos desconstruir a
narrativa tradicional e descobrir como essa era se conecta com a educação, a
crítica social e o poder da transformação.  **Em que contexto histórico o
Renascimento se inseriu?**   Como a educação se configurava na época? E como
essa educação se relaciona com as grandes transformações sociais e políticas que
marcaram essa era?  **Para além da simples memorização de nomes e datas, o
Renascimento nos convida a q

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

Device set to use mps:0


Intenção detectada pelo NLU: Análise de Exemplo

MODELO: google/gemma-3-4b-it
PERGUNTA (Análise de Exemplo): Com base no texto sobre a urbanização brasileira, quais foram os principais fatores que levaram ao crescimento desordenado das cidades?
--------------------
RESPOSTA GERADA:
Ok, vamos lá. Respondendo à pergunta do aluno sobre o crescimento desordenado
das cidades no Brasil, e aplicando os princípios da pedagogia de Paulo Freire e
da análise sociotécnica, como um especialista nessa área, proponho a seguinte
resposta:  “Essa pergunta é um ótimo ponto de partida para a gente começar a
pensar sobre o que realmente está acontecendo em nossas cidades, não apenas como
um fato, mas como um processo social complexo.   Em vez de simplesmente listar
os fatores que o texto cita – como a industrialização, a migração rural, a falta
de planejamento urbano – eu gostaria que você refletisse um pouco mais.  **O que
mais chamou sua atenção nesse trecho sobre o crescimento das cidades? Qual
aspecto

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use mps:0


Intenção detectada pelo NLU: Conceitual

MODELO: google/gemma-3-4b-it
PERGUNTA (Comparativo): Qual a principal diferença entre os sistemas presidencialista e parlamentarista?
--------------------
RESPOSTA GERADA:
Olá! Excelente pergunta! É fundamental entendermos as diferenças entre sistemas
políticos como o presidencialista e o parlamentarista, e isso nos abre um leque
de possibilidades para refletir sobre poder, participação e, claro, a construção
do conhecimento. Mas, em vez de apenas te dizer qual é a diferença, vamos
pensar: **em quais contextos sociais e políticos essas diferenças se manifestam
de forma mais evidente?**  Em quais situações a forma como o poder é exercido
impacta diretamente a vida das pessoas?  Acho que essa questão nos leva
diretamente a um tema gerador: **a desigualdade social e a participação
política**.  Pensemos, por exemplo, em um bairro onde a vida das pessoas é muito
afetada pelas decisões tomadas no governo. Como as diferentes formas de governo
influenci